<a href="https://colab.research.google.com/github/tenghjjp-blip/test/blob/main/%E6%A4%8D%E7%94%B0%E3%82%B5%E3%83%96%E3%82%BC%E3%83%9F_24_%E5%95%8F%E9%A1%8C9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 問題９

1月から3月に生まれたいわゆる早生まれの人は、4月から6月に生まれた人に比較して、同学年において他の人に比べて年齢が半年から1年少ないために、不利だといわれている。このことを発達心理学では、相対年齢効果という。

この問題では、TINSS（IrendsinMathematicsandScienceStudy，「国際数学・理科教育動向調査」）のデータを用いて、中学2年における、生まれ月が数学の成績に与える影響について分析しよう。


##手順
①データの読み込み・データの中身の確認

②欠損値の確認

③データ型の確認

④回帰分析

（多重共線性の確認）

In [1]:
from scipy.stats import t, f
import numpy as np
import pandas as pd
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
# import wooldridge

In [9]:
#①データの読み込み
df = pd.read_excel("/content/timss.xlsx")
df.head()
# データはドライブからダウンロード
# ファイルのパス名（/content/timss.xlsx)は自身の環境に合わせて変更して。
# パス名とはファイルの位置のこと
# ファイルのパス名は左のフォルダっぽいアイコンを推して、「timss.xlsx」を見つけよう
# 詳しくは、次のリンクをチェック　https://interface.cqpub.co.jp/ail01/

,idschool,gender,numpeople,mathscore,sciencescore,agese_q2,agese_q3,agese_q4,comu_1,comu_2,...,mothereduc_3,mothereduc_4,mothereduc_5,mothereduc_6,fathereduc_1,fathereduc_2,fathereduc_3,fathereduc_4,fathereduc_5,fathereduc_6
0,1,girl,6,131.718887,142.569763,0,0,1,1,0,...,0,0,0,1,0,0,0,0,0,1
1,1,girl,4,145.550461,159.221176,0,1,0,1,0,...,0,1,0,0,0,0,0,1,0,0
2,1,girl,5,139.960510,140.885910,0,1,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,1,girl,3,145.478806,164.459839,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,1,boy,4,164.112015,160.343750,1,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1


## (a)
数学の成績(mathscore)を、生まれた四半期のダミー変数（agese_q2, agese_q3, agese_q4）に（定数項を含んで）回帰して、早生まれの人の成績が、4月から6月に生まれた人に比べて低いかどうかを検定しなさい。

**どの変数で評価すれば良いかを、検討しよう**

In [10]:
df.describe()

,idschool,mathscore,sciencescore,agese_q2,agese_q3,agese_q4,comu_1,comu_2,comu_3,comu_4,...,mothereduc_3,mothereduc_4,mothereduc_5,mothereduc_6,fathereduc_1,fathereduc_2,fathereduc_3,fathereduc_4,fathereduc_5,fathereduc_6
count,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,...,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000,4536.000000
mean,73.182099,149.968223,149.996500,0.272046,0.247575,0.229056,0.243386,0.395282,0.111772,0.160494,...,0.181217,0.166446,0.007275,0.291446,0.037919,0.266975,0.053792,0.283730,0.018519,0.339065
std,42.221116,9.932715,9.974685,0.445062,0.431651,0.420272,0.429174,0.488965,0.315121,0.367104,...,0.385240,0.372522,0.084993,0.454479,0.191021,0.442428,0.225631,0.450857,0.134832,0.473444
min,1.000000,107.280724,99.631523,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37.000000,143.257156,143.785873,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,73.000000,149.563766,149.959991,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,108.000000,156.443726,156.508316,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
max,148.000000,178.946915,194.114349,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## (b)
間（a）の回帰式に、

1.   地域の大きさを表すダミー変数(comu_1 から comu_5)
2.   コンピュータが家庭にあるかどうかを示すダミー変数(computer)
3. 家庭の人数を表す変数(numpeople)
4. 母親の学歴を表すダミー変数(mothereduc_1 から mothereduc_5)
5. 父親の学歴を表すダミー変数(fathereduc_1 から fathereduc_5)


を、説明変数に加えたモデルを推定しなさい。


その結果をもとに、早生まれの人の成績が、4月から6月に生まれた人に比べて低いかどうかを検定しなさい

In [11]:
df.dtypes

,0
idschool,int64
gender,object
numpeople,object
mathscore,float64
sciencescore,float64
agese_q2,int64
agese_q3,int64
agese_q4,int64
comu_1,int64
comu_2,int64


## (c)
数学の成績の代わりに、理科の成績(sciencescore)を用いて、問（a）,(b)の分析を行いなさい。数学と理科では、結果が異なるかどうかを議論しなさい。

### (c)-(a)


### (c)-(b)